In [ ]:
!pip install -q fastapi uvicorn chromadb llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface

In [ ]:
import os

# 1. Create a folder named 'company_docs' inside Colab
os.makedirs("company_docs", exist_ok=True)

# 2. Write a mock internal troubleshooting manual into the folder
manual_content = """
# Internal Engineering Runbook: Database & Storage Operations

## Error: Database Connection Timeout (Postgres Port 5432)
- **Problem**: Applications cannot talk to the database because idle background connections are clogging up the server pipeline.
- **Solution**: Open the terminal and run the command: `SELECT pg_terminate_backend(pid) FROM pg_stat_activity WHERE state = 'idle';`. This safely clears stale traffic loops.

## Error: AWS S3 Bucket Access Denied (Code 403)
- **Problem**: The storage bucket security settings are blocking permissions because the IAM access key is missing a valid policy.
- **Solution**: Open the AWS Console, navigate to the storage bucket policy tab, and apply a permission schema that allows read/write access to your specific API worker role.

## Error: Memory Leak Warning in API Routing
- **Problem**: Python loops are staying open in the background, consuming all RAM space.
- **Solution**: Wrap all live data streams inside python context managers (`with` statements) so they close automatically.
"""

with open("company_docs/troubleshooting_guide.txt", "w") as f:
    f.write(manual_content.strip())

print("✅ Folder 'company_docs' created and mock runbook file saved successfully!")

✅ Folder 'company_docs' created and mock runbook file saved successfully!


In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# 1. Load the local HuggingFace AI model to handle meaning-to-vector math
print("⚡ Loading local AI embedding model... (this takes a few seconds)")
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Tell LlamaIndex to use this model for all upcoming math actions
Settings.embed_model = embed_model
Settings.llm = None  # We set the LLM to None for now to focus entirely on vector retrieval

# 2. Initialize a local, persistent ChromaDB database on disk
chroma_client = chromadb.PersistentClient(path="./chroma_db_vault")

# Create a storage table (collection) inside the database named 'secure_logs'
chroma_collection = chroma_client.get_or_create_collection("secure_logs")

# 3. Connect LlamaIndex directly to our ChromaDB database storage layer
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("✅ ChromaDB Core Engine successfully initialized on T4 GPU runtime!")

⚡ Loading local AI embedding model... (this takes a few seconds)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LLM is explicitly disabled. Using MockLLM.
✅ ChromaDB Core Engine successfully initialized on T4 GPU runtime!


In [ ]:
# 1. Read all files inside the 'company_docs' folder
print("📂 Scanning company_docs directory...")
documents = SimpleDirectoryReader("./company_docs").load_data()

# 2. Process documents, convert them to vectors, and store them directly in ChromaDB
print("📐 Converting text paragraphs to vector arrays and saving to ChromaDB...")
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

print("🎉 Knowledge base is successfully prepared and locked inside ChromaDB!")

📂 Scanning company_docs directory...
📐 Converting text paragraphs to vector arrays and saving to ChromaDB...
🎉 Knowledge base is successfully prepared and locked inside ChromaDB!


In [ ]:
# Create a search engine handle from our index map
search_engine = index.as_retriever(similarity_top_k=1)

# Type a test question that uses completely different words from our text file
test_question = "How do I fix connection dropped errors on database port 5432?"

print(f"🕵️ Developer asks: '{test_question}'\n")

# Run the mathematical search
retrieved_nodes = search_engine.retrieve(test_question)

# Print out the exact text chunk that ChromaDB found sitting closest to our question math
if retrieved_nodes:
    best_match = retrieved_nodes[0].node.get_content()
    match_score = retrieved_nodes[0].score
    print(f"🎯 Smart Database found a match! (Confidence Math Score: {match_score:.4f})")
    print("-" * 60)
    print(best_match)
    print("-" * 60)
else:
    print("❌ No matching solution found.")

🕵️ Developer asks: 'How do I fix connection dropped errors on database port 5432?'

🎯 Smart Database found a match! (Confidence Math Score: 0.5524)
------------------------------------------------------------
# Internal Engineering Runbook: Database & Storage Operations

## Error: Database Connection Timeout (Postgres Port 5432)
- **Problem**: Applications cannot talk to the database because idle background connections are clogging up the server pipeline.
- **Solution**: Open the terminal and run the command: `SELECT pg_terminate_backend(pid) FROM pg_stat_activity WHERE state = 'idle';`. This safely clears stale traffic loops.

## Error: AWS S3 Bucket Access Denied (Code 403)
- **Problem**: The storage bucket security settings are blocking permissions because the IAM access key is missing a valid policy.
- **Solution**: Open the AWS Console, navigate to the storage bucket policy tab, and apply a permission schema that allows read/write access to your specific API worker role.

## Error

In [ ]:
!pip install pyngrok

In [ ]:
import threading
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok

# 1. Apply our background loop patch
nest_asyncio.apply()

# 2. Initialize the FastAPI Web Service Application
app = FastAPI(
    title="SecureOps AI API Node",
    description="Enterprise Compliance & Troubleshooting Vector Search Core",
    version="1.0.0"
)

# 3. Define the incoming request payload structure
class UserQuery(BaseModel):
    question: str

# 4. Define our base web routes
@app.get("/")
def home():
    return {"status": "Online", "engine": "ChromaDB-LlamaIndex-Core"}

@app.post("/api/v1/search")
async def ask_secure_ops(payload: UserQuery):
    try:
        search_engine = index.as_retriever(similarity_top_k=1)
        nodes = search_engine.retrieve(payload.question)

        if not nodes:
            return {"query": payload.question, "solution": "No matching runbook found."}

        best_match_text = nodes[0].node.get_content()
        match_score = nodes[0].score

        return {
            "query": payload.question,
            "solution": best_match_text,
            "confidence_score": float(match_score),
            "security_status": "Passed (Local Retrieval Execution)"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 5. Connect your free Ngrok account token to build the public internet proxy link
# 5. Securely pull your token from the Colab Vault behind the scenes
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)
    print("🔒 Security Verification: Successfully retrieved token from the vault!")
except Exception as e:
    print("⚠️ Security Configuration Error: Did you add 'NGROK_TOKEN' to your Colab Key Vault?")

# Clean up old processes and establish the secure tunnel connection
ngrok.kill()
public_tunnel = ngrok.connect(8000)

print("\n" + "="*70)
print(f"🌍 LIVE PUBLIC SWAGGER DOCUMENTATION URL:")
print(f"👉 {public_tunnel.public_url}/docs")
print("="*70 + "\n")
def run_server():
    # ✨ CHANGED host="0.0.0.0" TO host="127.0.0.1" HERE!
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("🚀 Backend engine has officially answered the gate on 127.0.0.1:8000!")

⚠️ Security Configuration Error: Did you add 'NGROK_TOKEN' to your Colab Key Vault?

🌍 LIVE PUBLIC SWAGGER DOCUMENTATION URL:
👉 https://turbulent-hatchback-snowsuit.ngrok-free.dev/docs

🚀 Backend engine has officially answered the gate on 127.0.0.1:8000!


In [ ]:
import threading
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from pyngrok import ngrok
from google.colab import userdata

# 1. Apply background network loop patch for Jupyter runtimes
nest_asyncio.apply()

# 2. Initialize the FastAPI Web Service Application Engine
app = FastAPI(
    title="SecureOps AI API Node",
    description="Enterprise Compliance & Troubleshooting Vector Search Core",
    version="1.0.0"
)

# 3. Define the structural incoming request payload rule
class UserQuery(BaseModel):
    question: str

# 4. BACKEND ROUTE: Run the semantic vector similarity search
@app.post("/api/v1/search")
async def ask_secure_ops(payload: UserQuery):
    try:
        # Connect to your active LlamaIndex database setup
        search_engine = index.as_retriever(similarity_top_k=1)
        nodes = search_engine.retrieve(payload.question)

        if not nodes:
            return {"query": payload.question, "solution": "No matching runbook protocols found."}

        best_match_text = nodes[0].node.get_content()
        match_score = nodes[0].score

        return {
            "query": payload.question,
            "solution": best_match_text,
            "confidence_score": float(match_score),
            "security_status": "Passed (Local Retrieval Execution)"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 5. FRONTEND ROUTE: Serve the upgraded multi-tab interactive platform dashboard
@app.get("/dashboard", response_class=HTMLResponse)
def serve_dashboard():
    return """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>SecureOps AI — Management Center</title>
        <script src="https://cdn.tailwindcss.com"></script>
        <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
        <style>
            body { font-family: 'Inter', sans-serif; }
            .tab-content { display: none; }
            .tab-content.active { display: block; }
        </style>
    </head>
    <body class="bg-slate-900 text-slate-100 min-h-screen flex flex-col">

        <nav class="border-b border-slate-800 bg-slate-950/50 backdrop-blur px-8 py-4 flex items-center justify-between sticky top-0 z-50">
            <div class="flex items-center space-x-3">
                <div class="h-3 w-3 rounded-full bg-emerald-500 animate-pulse"></div>
                <span class="font-bold tracking-tight text-lg text-white">SECUREOPS <span class="text-emerald-400">AI</span></span>
            </div>
            <div class="flex space-x-1 bg-slate-900 p-1 rounded-lg border border-slate-800">
                <button onclick="switchTab('search-tab')" id="btn-search-tab" class="tab-btn px-4 py-1.5 rounded-md text-xs font-medium transition-all bg-slate-800 text-white">Search Core</button>
                <button onclick="switchTab('about-tab')" id="btn-about-tab" class="tab-btn px-4 py-1.5 rounded-md text-xs font-medium transition-all text-slate-400 hover:text-slate-200">System Overview</button>
                <button onclick="switchTab('trouble-tab')" id="btn-trouble-tab" class="tab-btn px-4 py-1.5 rounded-md text-xs font-medium transition-all text-slate-400 hover:text-slate-200">Platform Diagnostic</button>
            </div>
            <div class="hidden md:flex items-center space-x-4 text-xs text-slate-400 font-medium bg-slate-900 border border-slate-800 px-3 py-1.5 rounded-md">
                <span>NODE STATUS: SECURE</span>
            </div>
        </nav>

        <main class="flex-1 max-w-5xl w-full mx-auto p-6 md:p-12">

            <div id="search-tab" class="tab-content active space-y-8">
                <div>
                    <h1 class="text-3xl font-bold text-white tracking-tight">Enterprise Infrastructure Search</h1>
                    <p class="text-slate-400 mt-2 text-sm">Query localized compliance manuals and operational runbooks using mathematical AI matching.</p>
                </div>

                <div class="grid grid-cols-1 md:grid-cols-3 gap-4">
                    <div class="bg-slate-950 border border-slate-800 rounded-xl p-5">
                        <div class="text-xs font-semibold text-slate-500 uppercase tracking-wider">Mean Time To Resolution</div>
                        <div class="text-2xl font-bold text-emerald-400 mt-1 font-mono">-84% Reduction</div>
                        <div class="text-xs text-slate-400 mt-1">Instant localized retrieval instead of manual searching.</div>
                    </div>
                    <div class="bg-slate-950 border border-slate-800 rounded-xl p-5">
                        <div class="text-xs font-semibold text-slate-500 uppercase tracking-wider">Data Operational Integrity</div>
                        <div class="text-2xl font-bold text-white mt-1 font-mono">100% Locked</div>
                        <div class="text-xs text-slate-400 mt-1">Zero public leakages. Runbooks stored behind secure firewalls.</div>
                    </div>
                    <div class="bg-slate-950 border border-slate-800 rounded-xl p-5">
                        <div class="text-xs font-semibold text-slate-500 uppercase tracking-wider">AI Retrieval Framework</div>
                        <div class="text-2xl font-bold text-white mt-1 font-mono">Dense Vectors</div>
                        <div class="text-xs text-slate-400 mt-1">Tracks contextual engineering meaning over rigid keywords.</div>
                    </div>
                </div>

                <div class="bg-slate-950 border border-slate-800 rounded-xl p-2 flex items-center shadow-2xl focus-within:border-emerald-500/50 transition-all duration-200">
                    <input
                        type="text"
                        id="userQuestion"
                        placeholder="Describe the system fault or paste a log trace phrase (e.g., AWS S3 403 or DB connection timeout)..."
                        class="w-full bg-transparent border-0 outline-none px-4 py-3 text-slate-200 placeholder-slate-500 text-sm"
                    >
                    <button
                        onclick="submitSearchQuery()"
                        class="bg-emerald-500 hover:bg-emerald-600 text-slate-950 font-semibold text-xs uppercase tracking-wider px-6 py-3 rounded-lg transition-all"
                    >
                        Analyze
                    </button>
                </div>

                <div id="resultsCard" class="hidden bg-slate-950 border border-slate-800 rounded-xl p-6 shadow-2xl transition-all duration-300">
                    <div class="flex items-center justify-between border-b border-slate-800 pb-4 mb-4">
                        <span class="text-xs font-semibold bg-emerald-500/10 text-emerald-400 border border-emerald-500/20 px-2.5 py-1 rounded-md uppercase tracking-wider">Verified Resolution Protocol</span>
                        <div class="text-xs text-slate-500">
                            Match Confidence: <span id="confidenceMetric" class="text-slate-300 font-mono font-medium">0.00%</span>
                        </div>
                    </div>
                    <div class="text-slate-300 text-sm leading-relaxed whitespace-pre-line font-normal" id="solutionContainer"></div>
                </div>
            </div>

            <div id="about-tab" class="tab-content space-y-8">
                <div>
                    <h1 class="text-3xl font-bold text-white tracking-tight">System Documentation & Architecture</h1>
                    <p class="text-slate-400 mt-2 text-sm">Understand the operational mechanics and corporate data rules of this node.</p>
                </div>

                <div class="grid grid-cols-1 md:grid-cols-2 gap-8">
                    <div class="space-y-4">
                        <h3 class="text-lg font-semibold text-emerald-400 border-b border-slate-800 pb-2">What is SecureOps AI?</h3>
                        <p class="text-slate-300 text-sm leading-relaxed">
                            SecureOps AI is an on-premise Retrieval-Augmented Generation engine designed to safely process, index, and query confidential enterprise documentation, operations logs, and technical infrastructure frameworks without exposing assets to external providers.
                        </p>
                        <h3 class="text-lg font-semibold text-emerald-400 border-b border-slate-800 pb-2 pt-2">Operational Guidelines</h3>
                        <ol class="list-decimal list-inside space-y-2 text-sm text-slate-300">
                            <li><strong class="text-white">Input:</strong> Enter plain-text infrastructure alerts straight into the primary dashboard console.</li>
                            <li><strong class="text-white">Mapping:</strong> The system calculates semantic similarity weights using localized transformers.</li>
                            <li><strong class="text-white">Resolution:</strong> Read the extracted verified structural card details to resolve system anomalies.</li>
                        </ol>
                    </div>

                    <div class="bg-slate-950 border border-slate-800 rounded-xl p-6 space-y-4">
                        <h3 class="text-md font-semibold text-white tracking-tight">System Architecture Safeguards</h3>
                        <div class="space-y-4">
                            <div class="flex items-start space-x-3">
                                <div class="bg-emerald-500/10 text-emerald-400 p-2 rounded-lg text-xs font-bold font-mono">01</div>
                                <div>
                                    <h4 class="text-sm font-semibold text-slate-200">Air-Gapped Data Privacy</h4>
                                    <p class="text-xs text-slate-400 mt-0.5">Utilizes local huggingface transformers to compute embeddings on company servers. Intellectual assets never traverse the public internet.</p>
                                </div>
                            </div>
                            <div class="flex items-start space-x-3">
                                <div class="bg-emerald-500/10 text-emerald-400 p-2 rounded-lg text-xs font-bold font-mono">02</div>
                                <div>
                                    <h4 class="text-sm font-semibold text-slate-200">Contextual Ingestion Architecture</h4>
                                    <p class="text-xs text-slate-400 mt-0.5">Bypasses brittle keyword logic. ChromaDB segments files by technical context, ensuring relevant records match even if wording syntax varies.</p>
                                </div>
                            </div>
                        </div>
                    </div>
                </div>
            </div>

            <div id="trouble-tab" class="tab-content space-y-8">
                <div>
                    <h1 class="text-3xl font-bold text-white tracking-tight">Platform Diagnostics & Self-Healing</h1>
                    <p class="text-slate-400 mt-2 text-sm">Audit diagnostic status flags and resolve localized node operational errors.</p>
                </div>

                <div class="space-y-4 max-w-3xl">
                    <div class="bg-slate-950 border border-slate-800 rounded-xl p-5">
                        <h4 class="text-sm font-semibold text-white flex items-center justify-between">
                            <span>🛑 Query Execution Returns a "Communication Error" Alert Box</span>
                            <span class="text-xs text-amber-500 bg-amber-500/10 px-2 py-0.5 rounded font-mono border border-amber-500/20">Network Check</span>
                        </h4>
                        <p class="text-xs text-slate-400 mt-2 leading-relaxed">
                            <strong class="text-slate-300">Root Cause:</strong> The background execution thread processing requests inside your computing environment has timed out, or your proxy tunnel connection parameters expired.<br>
                            <strong class="text-slate-300">Resolution Protocol:</strong> Re-run this integrated all-in-one cell script to reboot active listening parameters on the network listener interface.
                        </p>
                    </div>

                    <div class="bg-slate-950 border border-slate-800 rounded-xl p-5">
                        <h4 class="text-sm font-semibold text-white flex items-center justify-between">
                            <span>🛑 AI Returns a 0.00% Match Confidence or Irrelevant Data Cards</span>
                            <span class="text-xs text-blue-500 bg-blue-500/10 px-2 py-0.5 rounded font-mono border border-blue-500/20">Ingestion Check</span>
                        </h4>
                        <p class="text-xs text-slate-400 mt-2 leading-relaxed">
                            <strong class="text-slate-300">Root Cause:</strong> The targeted problem statement context doesn't exist within any corporate manual or log template files parsed during database setup initialization.<br>
                            <strong class="text-slate-300">Resolution Protocol:</strong> Drag new plain-text files into the <code class="bg-slate-900 px-1 py-0.5 rounded border border-slate-800 text-slate-200">company_docs</code> filesystem directory and rebuild your indexing directory step.
                        </p>
                    </div>
                </div>
            </div>
        </main>

        <script>
            function switchTab(tabId) {
                document.querySelectorAll('.tab-content').forEach(el => el.classList.remove('active'));
                document.getElementById(tabId).classList.add('active');
                document.querySelectorAll('.tab-btn').forEach(btn => {
                    btn.classList.remove('bg-slate-800', 'text-white');
                    btn.classList.add('text-slate-400');
                });
                document.getElementById('btn-' + tabId).classList.add('bg-slate-800', 'text-white');
                document.getElementById('btn-' + tabId).classList.remove('text-slate-400');
            }

            async function submitSearchQuery() {
                const queryInput = document.getElementById('userQuestion').value;
                const resultsCard = document.getElementById('resultsCard');
                const solutionContainer = document.getElementById('solutionContainer');
                const confidenceMetric = document.getElementById('confidenceMetric');

                if(!queryInput.trim()) return alert("Please input an infrastructure query message.");

                solutionContainer.innerHTML = `<div class="text-slate-500 animate-pulse text-xs uppercase tracking-widest font-mono">Running vector similarity search...</div>`;
                resultsCard.classList.remove('hidden');

                try {
                    const response = await fetch('/api/v1/search', {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify({ question: queryInput })
                    });
                    const data = await response.json();
                    if(data.solution) {
                        solutionContainer.innerText = data.solution;
                        confidenceMetric.innerText = (data.confidence_score * 100).toFixed(2) + "%";
                    } else {
                        solutionContainer.innerText = "No resolution protocols matched.";
                        confidenceMetric.innerText = "0.00%";
                    }
                } catch (error) {
                    solutionContainer.innerText = "Error establishing communications with secure vector engine node.";
                }
            }
        </script>
    </body>
    </html>
    """

# 6. Securely retrieve credentials from Colab Secrets and run Ngrok tunnel proxy
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)
    ngrok.kill() # Instantly sweep away any dead or hung background proxy connections

    # Establish proxy line directly pointing to local loop gate
    public_tunnel = ngrok.connect(8000)
    print("\n" + "="*70)
    print(f"🔒 SECURE GLOBAL ACCESS NETWORK PORTALS INITIALIZED")
    print(f"👉 API SWAGGER PORTAL : {public_tunnel.public_url}/docs")
    print(f"👉 PRODUCT DASHBOARD  : {public_tunnel.public_url}/dashboard")
    print("="*70 + "\n")
except Exception as e:
    print(f"⚠️ Security Error: Could not execute tunnel launch. Ensure NGROK_TOKEN is toggled on in the Colab Secret Vault menu. Info: {e}")

# 7. Bind Uvicorn Server execution process loop to Localhost interface
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

# Boot up server pipeline within an isolated independent background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()


🔒 SECURE GLOBAL ACCESS NETWORK PORTALS INITIALIZED
👉 API SWAGGER PORTAL : https://turbulent-hatchback-snowsuit.ngrok-free.dev/docs
👉 PRODUCT DASHBOARD  : https://turbulent-hatchback-snowsuit.ngrok-free.dev/dashboard

